# Analysis Part 3 -- paper figures

Makes three IEEE-sized figures from the cleaned sweep results and saves PNG (300 dpi) + PDF to `MyDrive/MediSense-Cardio-Study/figures/`.
1. Heart-rate error vs damage level, per damage type (44 reliable records).
2. Heatmaps: how often the system wrongly said 'normal' (false reassurance) and how often it gave no answer.
3. The records whose pictures the digitizer fails to read most often.

Design rules: one measure per axis (no dual axes), one hue, thin marks, numbers written in the cells so nothing depends on color alone. Runs on CPU.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
D = '/content/drive/MyDrive/MediSense-Cardio-Study'
FIG = f'{D}/figures'; os.makedirs(FIG, exist_ok=True)
o = pd.read_csv(f'{D}/t3_hr_outcomes.csv')
pr = pd.read_csv(f'{D}/per_record_failures.csv')
o['diff_bpm'] = pd.to_numeric(o.diff_bpm, errors='coerce')
rel = o[o.reliable].copy()
print('tests:', len(o), '| reliable-record tests:', len(rel), '| reliable records:', rel.record.nunique())

# one hue (blue), neutral text/grid
BLUE, BLUE_D, INK, MUTE, GRID = '#2a78d6', '#1b4f91', '#0b0b0b', '#52514e', '#dcdcd8'
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 7, 'axes.edgecolor': MUTE, 'axes.labelcolor': INK,
    'xtick.color': MUTE, 'ytick.color': MUTE, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': 0.6, 'figure.dpi': 130, 'savefig.dpi': 300})
AX_ORDER = [('blur','Blur'), ('resolution','Low resolution'), ('rotate','Tilt (rotation)'),
            ('jpeg','JPEG compression'), ('gridcontrast','Faded grid'), ('brightness','Uneven lighting')]


## Figure 1 -- heart-rate error vs damage level


In [ ]:
clean = rel[rel.axis=='clean'].diff_bpm.dropna()
fig, axs = plt.subplots(2, 3, figsize=(7.16, 4.0), sharey=True, sharex=True)
stats = {}
for (key, title), ax in zip(AX_ORDER, axs.flat):
    g = rel[rel.axis==key]
    lv = [0] + sorted(g.level.unique())
    med = [clean.median()] + [g[g.level==l].diff_bpm.median() for l in lv[1:]]
    p90 = [clean.quantile(.9)] + [g[g.level==l].diff_bpm.quantile(.9) for l in lv[1:]]
    stats[key] = (med, p90)
    ax.axhline(clean.median(), color=GRID, lw=0.8, zorder=0)
    ax.plot(lv, p90, color=BLUE, lw=1.2, ls='--', marker='o', ms=3.5, mfc='white', mec=BLUE, mew=0.9, label='90th percentile (1 in 10 readings is worse)')
    ax.plot(lv, med, color=BLUE_D, lw=1.4, marker='o', ms=3.5, label='median (typical reading)')
    ax.set_title(title, fontsize=7.5, color=INK, loc='left', pad=3)
    ax.set_yscale('symlog', linthresh=1); ax.set_ylim(0, 40)
    ax.set_yticks([0, 1, 2, 5, 10, 20, 40]); ax.set_yticklabels(['0','1','2','5','10','20','40'])
    ax.set_xticks(range(6)); ax.grid(axis='y', color=GRID, lw=0.4)
    if key == 'resolution':
        ax.annotate('%.1f' % p90[-1], (5, p90[-1]), textcoords='offset points', xytext=(-4, 4), ha='right', fontsize=6.5, color=INK)
for ax in axs[:, 0]: ax.set_ylabel('heart-rate error (bpm)')
for ax in axs[1, :]: ax.set_xlabel('damage level (0 = clean)')
h, l = axs[0,0].get_legend_handles_labels()
fig.legend(h, l, loc='upper center', ncol=2, frameon=False, fontsize=6.8, bbox_to_anchor=(0.5, 1.02))
fig.text(0.01, -0.01, 'n = 44 records per point. Grey line = error on the clean picture. Log-style scale so small and large errors are both visible.', fontsize=6, color=MUTE)
fig.tight_layout(rect=(0, 0.0, 1, 0.94))
for ext in ('png', 'pdf'): fig.savefig(f'{FIG}/fig1_hr_error_by_damage.{ext}', bbox_inches='tight')
plt.show()
print('median / 90th pct by level (0=clean):')
for k, (m, p) in stats.items(): print(k.ljust(13), 'median', np.round(m, 2), '| p90', np.round(p, 2))


## Figure 2 -- wrongly reassured, and no answer at all


In [ ]:
risk = rel[(rel.truth != 'normal')]
n_risk = risk.record.nunique(); n_rel = rel.record.nunique()
levels = [1, 2, 3, 4, 5]
real_fr = (risk.outcome=='false_reassurance') & (risk.diff_bpm > 10)
border = int(((risk.outcome=='false_reassurance') & ~(risk.diff_bpm > 10)).sum())
risk = risk.assign(real=real_fr)
A = np.array([[int(risk[(risk.axis==k)&(risk.level==l)].real.sum()) for l in levels] for k, _ in AX_ORDER])
B = np.array([[int((rel[(rel.axis==k)&(rel.level==l)].outcome=='silent_drop').sum()) for l in levels] for k, _ in AX_ORDER])
def heat(ax, M, denom, title, vmax):
    cmap = matplotlib.colors.LinearSegmentedColormap.from_list('b', ['#f4f8fd', BLUE_D])
    ax.imshow(M / denom, cmap=cmap, vmin=0, vmax=vmax, aspect='auto')
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            f = M[i, j] / denom
            ax.text(j, i, f'{M[i, j]}/{denom}', ha='center', va='center', fontsize=6.5, color='white' if f > vmax * 0.55 else INK)
    ax.set_xticks(range(len(levels))); ax.set_xticklabels(levels)
    ax.set_yticks(range(len(AX_ORDER))); ax.set_yticklabels([t for _, t in AX_ORDER])
    ax.set_xlabel('damage level'); ax.set_title(title, fontsize=7.2, loc='left', color=INK)
    ax.tick_params(length=0)
    for s in ax.spines.values(): s.set_visible(False)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(7.16, 2.9), gridspec_kw={'wspace': 0.42})
heat(a1, A, n_risk, f"Wrongly told 'normal' (reading off >10 bpm)\nof the {n_risk} records whose true rate is slow or fast", 0.5)
heat(a2, B, n_rel, f'No answer: picture unreadable\nof the {n_rel} reliable records', 0.15)
fig.text(0.01, -0.03, f'Each cell counts records. {border} further borderline flips (reading within 10 bpm of truth but across the 60/100 bpm line) are excluded from the left panel.', fontsize=6, color=MUTE)
for ext in ('png', 'pdf'): fig.savefig(f'{FIG}/fig2_false_reassurance_and_silent_drops.{ext}', bbox_inches='tight')
plt.show()
print('false reassurance (real) matrix rows=', [k for k,_ in AX_ORDER], 'cols=levels 1-5'); print(A)
print('silent drops matrix'); print(B)
print('borderline flips excluded:', border, '| at-risk records:', n_risk, '| reliable records:', n_rel)


## Figure 3 -- records that fail most often


In [ ]:
bad = pr[pr.n_unreadable > 0].sort_values('n_unreadable', ascending=True).tail(12)
lab = [f"{r} ({'abnormal' if a else 'normal'})" for r, a in zip(bad.record, bad.is_abnormal)]
fig, ax = plt.subplots(figsize=(3.5, 3.0))
ax.barh(lab, bad.n_unreadable, color=BLUE, height=0.62)
for y, v in enumerate(bad.n_unreadable): ax.text(v + 0.4, y, f'{int(v)}', va='center', fontsize=6.5, color=INK)
ax.axvline(31, color=GRID, lw=0.8); ax.set_xlim(0, 34)
ax.set_xlabel('unreadable versions out of 31'); ax.tick_params(axis='y', length=0)
ax.set_title('Records that fail most often (top 12 of 50)', fontsize=7.2, loc='left', color=INK)
ax.spines['left'].set_visible(False)
fig.tight_layout()
for ext in ('png', 'pdf'): fig.savefig(f'{FIG}/fig3_unreadable_records.{ext}', bbox_inches='tight')
plt.show()
print(bad[['record','is_abnormal','n_unreadable']].to_string(index=False))
print('files in figures/:', sorted(os.listdir(FIG)))
